In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from main import CUSTOMS_2015_REF
from graphs import plot_pivot_heatmap, plot_top10_bar
from src.pivot import build_pivot_table, pivot_interior_sum
from src.processor import generate_grouped_summary, generate_grouped_two_summary
from top10 import top10
from validation import AuditLogger, PipelineValidator

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

Matplotlib is building the font cache; this may take a moment.


In [2]:
input_file = Path(config.INPUT_PATH).resolve()
raw_df = pd.read_csv(input_file, encoding="latin1", low_memory=False)

audit = AuditLogger()
validator = PipelineValidator()

raw_rows, raw_cols = len(raw_df), raw_df.shape[1]
raw_measure_sum = float(raw_df[config.MEASURE_COL].sum(min_count=1))

print(f"Loaded {raw_rows:,} rows and {raw_cols} columns from {input_file.name}")
raw_df.head()

Loaded 2,236,612 rows and 30 columns from 2015.csv


,uid,ty,tq,tm,entry,hscode,goodsdescription,p,q,m_fob,...,vatbase,vatpaid,othertax,finesandpenalties,dutiestaxes,prefcode,countryorigin_iso3,countryexport_iso3,subport,port
0,201501 00000001,2015,2015q1,2015m1,C,15119090000,RBD PALM OLEIN IN BULK,0.64,"2,999,325.00","1,930,315.60",...,90638835,10876660,NaN,NaN,10876660,AFTA,MYS,MYS,Sub-Port of Dumaguete,Port of Cebu
1,201501 00000002,2015,2015q1,2015m1,C,72104990000,PRIME HOT DIPPED GALVANIZED STEEL SHEET IN C,0.63,"532,480.00","336,527.38",...,15884083,1906090,NaN,NaN,1906090,ACFTA,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
2,201501 00000003,2015,2015q1,2015m1,C,72104990000,PRIME HOT DIPPED GALVANIZED STEEL SHEET IN C,0.63,"1,779,450.00","1,115,715.50",...,53064898,6367787,NaN,NaN,6367787,ACFTA,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
3,201501 00000004,2015,2015q1,2015m1,C,72139100000,HOT ROLLED WIRE ROD SWRY11 6.5MM,0.47,"1,525,249.00","722,968.00",...,34664620,4159754,NaN,NaN,4501570,NaN,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
4,201501 00000005,2015,2015q1,2015m1,C,72163300000,STEEL STRUCTURE,1.20,"134,305.20","161,469.44",...,7555345,906641,NaN,NaN,906641,AKFTA,KOR,KOR,Harbour Centre Port Terminal Inc,Port of Manila


In [4]:
filtered_df = raw_df.copy()

filters = config.FILTERS or {}
for col, allowed in filters.items():
    filtered_df = filtered_df[filtered_df[col].isin(allowed) & filtered_df[col].notna()]

selected_rows = len(filtered_df)
print(f"Selected: {selected_rows:,} rows out of {raw_rows:,}")
filtered_df.head()

Selected: 2,236,612 rows out of 2,236,612


,uid,ty,tq,tm,entry,hscode,goodsdescription,p,q,m_fob,...,vatbase,vatpaid,othertax,finesandpenalties,dutiestaxes,prefcode,countryorigin_iso3,countryexport_iso3,subport,port
0,201501 00000001,2015,2015q1,2015m1,C,15119090000,RBD PALM OLEIN IN BULK,0.64,"2,999,325.00","1,930,315.60",...,90638835,10876660,NaN,NaN,10876660,AFTA,MYS,MYS,Sub-Port of Dumaguete,Port of Cebu
1,201501 00000002,2015,2015q1,2015m1,C,72104990000,PRIME HOT DIPPED GALVANIZED STEEL SHEET IN C,0.63,"532,480.00","336,527.38",...,15884083,1906090,NaN,NaN,1906090,ACFTA,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
2,201501 00000003,2015,2015q1,2015m1,C,72104990000,PRIME HOT DIPPED GALVANIZED STEEL SHEET IN C,0.63,"1,779,450.00","1,115,715.50",...,53064898,6367787,NaN,NaN,6367787,ACFTA,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
3,201501 00000004,2015,2015q1,2015m1,C,72139100000,HOT ROLLED WIRE ROD SWRY11 6.5MM,0.47,"1,525,249.00","722,968.00",...,34664620,4159754,NaN,NaN,4501570,NaN,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
4,201501 00000005,2015,2015q1,2015m1,C,72163300000,STEEL STRUCTURE,1.20,"134,305.20","161,469.44",...,7555345,906641,NaN,NaN,906641,AKFTA,KOR,KOR,Harbour Centre Port Terminal Inc,Port of Manila


In [5]:
# Generate grouped summaries using src modules
grouped_df = generate_grouped_summary(
    filtered_df,
    group_col=config.GROUP_COL_ONE,
    measure_col=config.MEASURE_COL,
    output_path=str(Path(config.OUTPUT_DIR) / "grouped.csv"),
)

grouped_two_df = generate_grouped_two_summary(
    filtered_df,
    group_cols=config.GROUP_COLS_TWO,
    measure_col=config.MEASURE_COL,
    output_path=str(Path(config.OUTPUT_DIR) / "grouped_two.csv"),
)

# Build pivot table using src/pivot.py
pivot_df = build_pivot_table(
    filtered_df,
    index_col=config.GROUP_COLS_TWO[0],
    columns_col=config.GROUP_COLS_TWO[1],
    value_col=config.MEASURE_COL,
    output_path=str(Path(config.OUTPUT_DIR) / "pivot.csv"),
)

p_int_sum = pivot_interior_sum(pivot_df, margin_label="Total")
print(f"Pivot Interior Sum Validation: PHP {p_int_sum:,.2f}")
pivot_df

Pivot Interior Sum Validation: PHP 3,587,267,375,257.00


tq,2015q1,2015q2,2015q3,2015q4,Total
countryorigin_iso3,,,,,
AFG,0,0,567713,20833,588546
AGO,0,0,11752842,0,11752842
AIA,0,0,1429763,4690247,6120010
ALB,0,138212,947695,3269863,4355770
AND,0,1551437,4748724,9928679,16228840
...,...,...,...,...,...
ZAF,652523252,592767299,945248670,653652723,2844191944
ZAR,0,65285,182729512,3613885,186408682
ZMB,8292766,26435395,32348410,75552495,142629066


In [7]:
!jupyter nbconvert --to html analysis.ipynb

[NbConvertApp] Converting notebook analysis.ipynb to html
[NbConvertApp] Writing 300822 bytes to analysis.html
